# CombiCone — triage a combinatorial perturbation screen end-to-end

**Audience:** a screening / target-discovery team deciding *which combinations to run* and
*which measured combinations are genuinely emergent*.

A combinatorial screen explodes super-linearly: 100 single perturbations imply ~5,000 pairs
and ~160,000 triples. You cannot measure them all, and most combinations are **additive** —
they do nothing the two singles didn't already do, so the well is wasted. The decision that
costs money is: **which combinations produce a cell state the singles cannot reach on their
own?**

This notebook shows both halves of the CombiCone workflow on a small **synthetic** screen
with *planted* ground truth, so you can see the method recover what it should:

1. **Prospective triage** — rank unmeasured combinations from the single-gene effects alone.
2. **Certified emergence** — for a *measured* combination, emit a fail-closed, model-relative
   separator **and** a measurement-noise test, so a low signal-to-noise residual is never
   mistaken for biology.

> **Scope, stated up front.** "Unreachable" is *model-relative*: outside the non-negative cone
> of **these** measured single-gene effects under **this** metric — never a claim of biological
> impossibility. Every result object carries this in a `scope` field. This is a triage and
> hypothesis-ranking instrument, not a calibrated biological decision, a dose, or a target
> validation. It is deterministic and runs on synthetic data — no downloads, credentials, or
> `data/` directory required.

## Setup

In [1]:
import numpy as np

import combicone as cc
import reachability as rx

print("combicone entry points:", [x for x in cc.__all__ if not x.startswith("_")])

combicone entry points: ['EmergenceCertificate', 'TriageResult', 'triage_combinations', 'certify_emergence', 'fit_triage_model', 'single_effect_cosine', 'additive_gap']


## 1. A synthetic combinatorial screen with known ground truth

We simulate a legible screen: `N = 12` single-gene perturbations over `G = 60` genes. Each
single has a distinct sparse effect signature. We then construct nine *measured* double
perturbations of three known kinds:

- **emergent** (4): the additive expectation `A + B` **plus** a component orthogonal to the
  entire single-gene cone — a direction nothing in the library can reach.
- **additive** (4): exactly `A + B` — reachable from the singles, no emergence.
- **noise trap** (1): an *additive* (reachable) combination at very low signal magnitude. Its
  *normalized* residual is inflated by measurement noise and will look moderately "unreachable"
  — the trap a raw residual ranking falls into.

Measurement noise is **absolute** (the same per-gene standard error everywhere), which is what
makes the low-magnitude trap noise-dominated — exactly the regime seen in real screens.

In [2]:
rng = np.random.default_rng(20260719)
G, N = 60, 12
names = [f"P{i:02d}" for i in range(N)]
singles = np.zeros((N, G))

# Emergent-source singles P00..P07: each gets a DISJOINT 5-gene block, so any pair of
# them points in near-orthogonal directions (low single-single cosine). Low cosine is
# the training-free triage signal, and these are the pairs we will plant as emergent.
for i in range(8):
    singles[i, i*5 : i*5 + 5] = rng.normal(1.0, 0.3, size=5)

# Additive-source singles P08..P11: share a common core support, so they are mutually
# SIMILAR (high cosine) -> their combinations are additive / reachable.
core = np.arange(40, 53)
for i in range(8, 12):
    singles[i, core] = np.abs(rng.normal(1.0, 0.15, size=len(core)))

singles *= rng.uniform(0.8, 1.4, size=(N, 1))
idx = {n: i for i, n in enumerate(names)}
ABS_NOISE = 0.05  # per-gene measurement SE (what a split-half would estimate)

def measured_double(a, b, geometry, mag, seed):
    r = np.random.default_rng(seed)
    base = singles[idx[a]] + singles[idx[b]]
    if geometry == "emergent":
        off = np.zeros(G)
        off[r.choice(G, 4, replace=False)] = r.normal(2.0, 0.3, 4)
        Q, _ = np.linalg.qr(singles.T)      # orthonormal basis for the single-gene span
        off = off - Q @ (Q.T @ off)         # push OUT of that span -> genuinely unreachable
        base = base + 1.4 * off
    return base * mag + r.normal(0, ABS_NOISE, size=G)

truth = {
    ("P00","P01"):"emergent", ("P02","P03"):"emergent",
    ("P04","P05"):"emergent", ("P06","P07"):"emergent",
    ("P08","P09"):"additive", ("P08","P10"):"additive",
    ("P09","P11"):"additive", ("P10","P11"):"additive",
    ("P08","P11"):"noise_trap",   # additive geometry, tiny magnitude -> noise-dominated
}
measured, noise_sd = {}, {}
for k, (pair, kind) in enumerate(truth.items()):
    geometry = "emergent" if kind == "emergent" else "additive"
    mag = 0.12 if kind == "noise_trap" else 1.0
    measured[pair] = measured_double(pair[0], pair[1], geometry, mag, seed=100 + k)
    noise_sd[pair] = np.full(G, ABS_NOISE)

print(f"singles: {singles.shape}   measured doubles: {len(measured)}")
print(f"{'combination':13s} {'planted':11s} {'||effect||':>10s} {'cos(A,B)':>9s}")
for pair, kind in truth.items():
    a, b = pair
    cab = cc.single_effect_cosine(singles[idx[a]], singles[idx[b]])
    print(f"{a+'+'+b:13s} {kind:11s} {np.linalg.norm(measured[pair]):10.2f} {cab:+9.2f}")

singles: (12, 60)   measured doubles: 9
combination   planted     ||effect||  cos(A,B)
P00+P01       emergent          6.26     +0.00
P02+P03       emergent          6.05     +0.00
P04+P05       emergent          6.43     +0.00
P06+P07       emergent          6.56     +0.00
P08+P09       additive          8.06     +0.99
P08+P10       additive          8.73     +0.98
P09+P11       additive          7.14     +0.98
P10+P11       additive          8.00     +0.99
P08+P11       noise_trap        1.11     +0.98


## 2. Prospective triage: which combinations to run first?

Before measuring a single double, `triage_combinations` ranks every candidate pair using only
the single-gene effects. The default training-free score is `-cos(effA, effB)`: two singles
that push transcription in **different** directions (low cosine) are the most likely to combine
into something off-cone. This is the strongest single feature validated on the real Norman
screen (top-20 precision 2.4x base rate vs the raw label; a labeled pilot can train
`fit_triage_model` for the noise-robust target).

We score **all** C(12, 2) = 66 candidate pairs and check where our four planted-emergent pairs
land.

In [3]:
# Triage the SHORTLIST you are considering: rank the nine candidate combos, run-first first.
shortlist = list(truth.keys())
tri = cc.triage_combinations(singles, names, shortlist)
run_order = [tri.pairs[i] for i in np.argsort(-tri.score)]

print(f"triage model: {tri.model}")
print("prioritized order for the shortlist (run first -> last):")
for r, pair in enumerate(run_order, 1):
    j = tri.pairs.index(pair)
    print(f"  {r}. {pair[0]}+{pair[1]:5s}  [{truth[pair]:10s}]  cos(A,B)={tri.cos_ab[j]:+.2f}")

# And across ALL C(12,2)=66 possible combinations: do the emergent pairs outrank the additive?
tri_all = cc.triage_combinations(singles, names)
rank = {p: r for p, r in zip(tri_all.pairs, tri_all.rank)}
em = [p for p, k in truth.items() if k == "emergent"]
ad = [p for p, k in truth.items() if k == "additive"]
correct = sum(rank[e] < rank[a] for e in em for a in ad)
print(f"\nacross all {len(tri_all.pairs)} candidate pairs: {correct}/{len(em)*len(ad)} "
      f"(emergent, additive) orderings correct  (separation AUC = {correct/(len(em)*len(ad)):.2f})")
print("emergent-pair ranks:", sorted(rank[p] for p in em))
print("additive-pair ranks:", sorted(rank[p] for p in ad))

triage model: training-free
prioritized order for the shortlist (run first -> last):
  1. P00+P01    [emergent  ]  cos(A,B)=+0.00
  2. P02+P03    [emergent  ]  cos(A,B)=+0.00
  3. P04+P05    [emergent  ]  cos(A,B)=+0.00
  4. P06+P07    [emergent  ]  cos(A,B)=+0.00
  5. P09+P11    [additive  ]  cos(A,B)=+0.98
  6. P08+P10    [additive  ]  cos(A,B)=+0.98
  7. P08+P11    [noise_trap]  cos(A,B)=+0.98
  8. P10+P11    [additive  ]  cos(A,B)=+0.99
  9. P08+P09    [additive  ]  cos(A,B)=+0.99

across all 66 candidate pairs: 16/16 (emergent, additive) orderings correct  (separation AUC = 1.00)
emergent-pair ranks: [1, 22, 39, 52]
additive-pair ranks: [62, 63, 65, 66]


From the single-gene effects alone — no measured double — triage puts all four planted-emergent
combinations at the top of the shortlist and every additive combination below them, and across
all 66 possible pairs every emergent pair outranks every additive pair (separation AUC = 1.00).
The dissimilar singles (low `cos(A, B)`) are the ones whose combination is most likely to leave
the cone. That is the pre-screen: measure the top-ranked candidates first.

Triage is **modest and honest**: on the real Norman screen the enrichment is ~2.4x over base
rate against the raw label (1.4x against the noise-robust label), a useful prior, not a solved
prediction. Use it to *order* a screen, not to skip measurement.

## 3. Certify a measured combination — with a measurement-noise test

Now suppose you have *measured* a combination. `certify_emergence` does two things:

1. **Point certificate** — projects the measured effect onto the single-gene cone. If it lands
   outside, it returns the **model-relative separator**: a vector proving the unreachable
   direction (`effects @ separator <= 0` for every single, `target @ separator > 0`).
2. **Noise test** — forms the honest null *"the target IS reachable and the residual is
   measurement noise"* by projecting onto the cone (best reachable point `f0`), then repeatedly
   adding fresh per-gene noise and re-projecting. It reports a bootstrap CI, a noise-injection
   p-value, and a **floor ratio** (observed residual / noise-only residual).

The verdict combines two increasingly strict bars:
- **bar (a)** p < 0.05 — the residual exceeds measurement noise;
- **bar (b)** floor ratio >= 1.9 — the effect size is well above the noise floor.

In [4]:
print(f"{'combination':13s} {'planted':11s} {'raw_resid':>9s} {'z':>7s} "
      f"{'p':>7s} {'floor':>6s}  verdict")
certs = {}
for pair, kind in truth.items():
    cert = cc.certify_emergence(singles, measured[pair], noise_sd=noise_sd[pair],
                                n_boot=300, seed=0)
    certs[pair] = cert
    print(f"{pair[0]+'+'+pair[1]:13s} {kind:11s} {cert.unreachable_fraction:9.3f} "
          f"{cert.z:7.1f} {cert.p_value:7.3g} {cert.floor_ratio:6.2f}  {cert.verdict}")

combination   planted     raw_resid       z       p  floor  verdict
P00+P01       emergent        0.875    64.4 0.00332   7.37  certified emergent (p=0.00332, 7.4x noise floor)
P02+P03       emergent        0.858    64.0 0.00332   7.44  certified emergent (p=0.00332, 7.4x noise floor)
P04+P05       emergent        0.839    71.0 0.00332   8.14  certified emergent (p=0.00332, 8.1x noise floor)
P06+P07       emergent        0.702    81.9 0.00332   9.07  certified emergent (p=0.00332, 9.1x noise floor)
P08+P09       additive        0.044     0.2   0.432   1.02  within measurement noise (p=0.432); do not call emergent
P08+P10       additive        0.043     0.6   0.269   1.06  within measurement noise (p=0.269); do not call emergent
P09+P11       additive        0.044    -1.2   0.897   0.88  within measurement noise (p=0.897); do not call emergent
P10+P11       additive        0.053     2.0  0.0266   1.20  emergent above noise but modest effect size (p=0.0266, only 1.2x noise floor)
P08+P11

Read the table by column, not by the raw residual:

- The four **planted-emergent** pairs certify cleanly: z ≈ 60-80, floor ratio ≈ 7-9x, both bars
  cleared.
- The **planted-additive** pairs stay near the noise floor (floor ratio ≈ 1). Two of them
  (e.g. `P08+P09`, `P08+P10`) are cleanly non-significant; but note that `P10+P11` scrapes across
  bar (a) at p ≈ 0.03 while its floor ratio is only ≈ 1.2x — **passing the p-value test does not
  make it emergent.** This is exactly why the verdict requires *both* bars: bar (a) (p < 0.05) is
  permissive, and bar (b) (floor ratio ≥ 1.9) is the discriminating effect-size test.
- The **noise trap** (`P08+P11`) is the sharpest version of the same lesson: its *raw* unreachable
  fraction (≈ 0.38) is in the same range as a real emergent pair, yet z ≈ 2 and floor ratio ≈ 1.2x,
  so it lands in the same "modest effect size" tier as `P10+P11` — **a raw-residual ranking would
  have flagged it as a top hit.**

This is precisely the signal-to-noise confound measured on the real Norman screen (raw residual
vs effect magnitude Spearman −0.56; only ~27% of doubles clear their own noise floor, even though
129/131 pass the p < 0.05 bar), and it is why the certificate — read on *both* bars — not the raw
residual, is what you act on.

## 4. Inspect one certificate in full

In [5]:
pair = ("P00", "P01")           # a planted-emergent combination
cert = certs[pair]
print(f"combination            : {pair[0]}+{pair[1]}")
print(f"geometry_status        : {cert.geometry_status}")
print(f"unreachable_fraction   : {cert.unreachable_fraction:.4f}")
print(f"90% bootstrap CI       : [{cert.ci_low:.4f}, {cert.ci_high:.4f}]")
print(f"noise-only residual    : {cert.noise_null_mean:.4f} +/- {cert.noise_null_sd:.4f}")
print(f"emergence z            : {cert.z:.1f}")
print(f"noise-injection p      : {cert.p_value:.4g}")
print(f"floor ratio            : {cert.floor_ratio:.2f}x")
print(f"verdict                : {cert.verdict}")
print(f"scope                  : {cert.scope}")

# The separator is a certificate: non-positive against every single, positive on the target.
sep = cert.separator
print(f"\nseparator shape        : {sep.shape}")
print(f"max_i (single_i . sep) : {float((singles @ sep).max()): .3e}   (<= 0 certifies)")
print(f"target . sep           : {float(measured[pair] @ sep): .3e}   (> 0)")

combination            : P00+P01
geometry_status        : outside_model_cone
unreachable_fraction   : 0.8754
90% bootstrap CI       : [0.8688, 0.8818]
noise-only residual    : 0.1188 +/- 0.0118
emergence z            : 64.4
noise-injection p      : 0.003322
floor ratio            : 7.37x
verdict                : certified emergent (p=0.00332, 7.4x noise floor)
scope                  : model-relative: unreachable = outside the non-negative cone of the supplied single-gene effects under the chosen metric; not a claim of biological impossibility

separator shape        : (60,)
max_i (single_i . sep) :  5.412e-16   (<= 0 certifies)
target . sep           :  3.007e+01   (> 0)


The separator is the artifact a virtual-cell predictor cannot give you: a **fail-closed
certificate of non-representability**. GEARS, scGPT, STATE, and CPA always emit *a* prediction;
CombiCone emits "this direction is not reachable from your library, and here is the vector that
proves it" — the honest input a triage decision needs.

## 5. Optional: train a triage model on a labeled pilot

If you can afford a small pilot where some combinations are measured and scored, `fit_triage_model`
fits a two-feature ridge (single-single cosine, leave-pair-out additive gap) to your labels. On
the real Norman screen this reaches leave-one-out Spearman ~0.43 (permutation p = 0.002) against
the noise-robust emergence label. Here we simply confirm the API round-trips and improves ranking
of the held-back pairs on our synthetic labels.

In [6]:
# Use the certified z of each measured double as the "pilot label".
labeled = {pair: float(certs[pair].z) for pair in truth}
model = cc.fit_triage_model(singles, names, labeled)
print("fitted ridge feature weights (cos_ab, gap):", np.round(model.coef_, 3))

# Re-rank the labeled pairs with the trained model and compare to the training-free score.
learned = cc.triage_combinations(singles, names, list(labeled.keys()), model=model)
free = cc.triage_combinations(singles, names, list(labeled.keys()))
truth_z = np.array([labeled[p] for p in learned.pairs])
print(f"corr(score, pilot label z): "
      f"training-free {np.corrcoef(free.score, truth_z)[0, 1]:+.2f}"
      f"   |   learned {np.corrcoef(learned.score, truth_z)[0, 1]:+.2f}")

fitted ridge feature weights (cos_ab, gap): [-11.136  11.108]
corr(score, pilot label z): training-free +0.99   |   learned +0.99


## 6. Where this maps onto real data

The workflow you just ran is the same one validated on the **Norman combinatorial CRISPRa
screen** (distributed file label `A549`; the canonical Norman 2019 screen is K562 — we report
it as file-label A549 and make no cell-line claim beyond that):

| Step | Synthetic here | Real Norman result |
|---|---|---|
| Cone geometry | 12 singles, 9 doubles | 105 single-gene atoms, 131 measured doubles, all outside the single-gene cone (fail-closed) |
| Noise test | trap demoted to ~1.2x floor | 129/131 pass p<0.05, only ~35/131 clear the 1.9x effect-size floor; raw residual is magnitude-confounded (-0.56), noise-aware z is not (+0.14) |
| Synergy recovery | planted emergence recovered | unsupervised, recovers DUSP9+MAPK1 phosphatase-on-substrate synergy; partial Spearman vs classic non-additivity 0.60 after controlling magnitude |
| Prospective triage | emergent pairs ranked to the top | top-20 precision 2.4x base rate (raw label), 1.4x (noise-robust); ridge LOO Spearman 0.43 |

To run it on your own screen, build single-gene effect vectors (`effect_dictionary.py` provides
a safe pooled condition-mean-minus-control adapter), estimate a per-gene noise SD from a random
cell split-half (`|t1 - t2| / 2`), and call the same two functions.

## Checks

In [7]:
# Determinism and the core invariants, as assertions (fail closed).
c1 = cc.certify_emergence(singles, measured[("P00","P01")], noise_sd=noise_sd[("P00","P01")], n_boot=100, seed=1)
c2 = cc.certify_emergence(singles, measured[("P00","P01")], noise_sd=noise_sd[("P00","P01")], n_boot=100, seed=1)
assert (c1.p_value, c1.z) == (c2.p_value, c2.z), "certify must be deterministic given a seed"

# Every planted-emergent double clears BOTH bars; no planted-additive double does.
for pair, kind in truth.items():
    c = certs[pair]
    both = (c.p_value < 0.05) and (c.floor_ratio >= 1.9)
    if kind == "emergent":
        assert both, f"{pair} should certify as emergent"
    else:
        assert not both, f"{pair} ({kind}) must NOT clear the strict emergence bar"

# The noise trap has a HIGH raw residual but is demoted by the noise test (that is the lesson).
trap = certs[("P08","P11")]
assert trap.unreachable_fraction > 0.2 and trap.floor_ratio < 2.0, "trap must be demoted by noise"

# Triage separates the classes perfectly on this screen.
assert correct == len(em) * len(ad), "triage should rank every emergent above every additive"

# The separator is a genuine certificate of non-representability.
sep = certs[("P00","P01")].separator
assert (singles @ sep).max() <= 1e-9 < (measured[("P00","P01")] @ sep), "separator must certify"
print("all checks passed")

all checks passed


## Next steps

- **Bring your own screen.** Replace `singles` / `measured` with your effect vectors and a
  split-half noise SD; the two calls are unchanged.
- **Read the certificate, not the raw residual.** Always report the p-value *and* the floor
  ratio, and say which bar you mean.
- **Triage is a prior, not a verdict.** Use the ranking to order wells; certify what you measure.
- See [`../README.md`](../README.md) for the full evidence, [`../combicone.py`](../combicone.py)
  for the API, and [`../results/findings.json`](../results/findings.json) for the canonical
  machine-readable results and claim boundaries.

*CombiCone emits geometry, a certificate, and a noise-calibrated verdict — not a biological
verdict, a dose, a recipe, or a wet-lab recommendation.*